In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define time axis (normalized to [-1, 1])
t = np.linspace(-1, 1, 500)

# Define first 4 Legendre polynomials BY HAND
P0 = np.ones_like(t)                    # P₀(t) = 1
P1 = t                                  # P₁(t) = t
P2 = (3*t**2 - 1) / 2                  # P₂(t) = (3t² - 1)/2
P3 = (5*t**3 - 3*t) / 2                # P₃(t) = (5t³ - 3t)/2

# Plot them
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

polynomials = [
    (P0, "P₀(t) = 1", "Constant (DC component)"),
    (P1, "P₁(t) = t", "Linear (trend)"),
    (P2, "P₂(t) = (3t² - 1)/2", "Quadratic (curvature)"),
    (P3, "P₃(t) = (5t³ - 3t)/2", "Cubic (S-curve)")
]

for idx, (poly, title, description) in enumerate(polynomials):
    axes[idx].plot(t, poly, linewidth=3, color=f'C{idx}')
    axes[idx].axhline(0, color='black', linewidth=0.5, alpha=0.3)
    axes[idx].axvline(0, color='black', linewidth=0.5, alpha=0.3)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_title(title, fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('t (normalized time)', fontsize=11)
    axes[idx].set_ylabel('Pₙ(t)', fontsize=11)
    axes[idx].text(0.05, 0.95, description, 
                   transform=axes[idx].transAxes,
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
                   fontsize=10)
    axes[idx].set_ylim(-1.5, 1.5)

plt.suptitle('The First 4 Legendre Polynomials', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("KEY OBSERVATIONS")
print("="*80)
print("\nP₀(t) = 1:")
print("  → Completely flat (captures average/DC value)")
print("\nP₁(t) = t:")
print("  → Straight line (captures linear trend)")
print("\nP₂(t) = (3t² - 1)/2:")
print("  → Parabola (captures curvature/acceleration)")
print("\nP₃(t) = (5t³ - 3t)/2:")
print("  → S-curve (captures inflection points)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import legendre
from scipy.linalg import lstsq

print("\n" + "="*80)
print("EXAMPLE: REPRESENTING A FALL SIGNAL")
print("="*80)

# Create time axis
n_samples = 200
t_raw = np.linspace(0, 1, n_samples)  # 0 to 1 second
t_norm = 2*t_raw - 1  # Normalize to [-1, 1] for Legendre

# Create a fall signal (acceleration)
fall_signal = np.zeros(n_samples)

# Phase 1: Walking (0-0.25s) - low, stable
fall_signal[:50] = 0.2 + 0.05*np.random.randn(50)

# Phase 2: Fall initiation (0.25-0.75s) - rapid increase
t_fall = t_raw[50:150]
fall_signal[50:150] = 0.2 + 3.0*(t_fall - 0.25)**2 + 0.1*np.random.randn(100)

# Phase 3: Impact (0.75-1.0s) - high, then decay
fall_signal[150:180] = 3.0 + 0.2*np.random.randn(30)
fall_signal[180:] = 3.0 * np.exp(-5*(t_raw[180:] - 0.9)) + 0.1*np.random.randn(20)

# Now approximate with Legendre polynomials
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

orders = [1, 3, 7, 15]  # Use 2, 4, 8, 16 polynomials

for idx, N in enumerate(orders):
    ax = axes[idx // 2, idx % 2]
    
    # Build Legendre basis matrix
    Phi = np.zeros((n_samples, N+1))
    for n in range(N+1):
        Pn = legendre(n)
        Phi[:, n] = Pn(t_norm)
    
    # Find coefficients by least squares
    coeffs, _, _, _ = lstsq(Phi, fall_signal)
    
    # Reconstruct
    reconstructed = Phi @ coeffs
    
    # Calculate error
    mse = np.mean((fall_signal - reconstructed)**2)
    
    # Plot
    ax.plot(t_raw, fall_signal, 'b-', linewidth=2, label='Actual Fall Signal', alpha=0.7)
    ax.plot(t_raw, reconstructed, 'r--', linewidth=2, label=f'Legendre Approx (N={N+1})')
    ax.axvspan(0, 0.25, alpha=0.1, color='green', label='Walking')
    ax.axvspan(0.25, 0.75, alpha=0.1, color='orange', label='Fall')
    ax.axvspan(0.75, 1.0, alpha=0.1, color='red', label='Impact')
    
    ax.set_title(f'Using {N+1} Legendre Polynomials (MSE={mse:.4f})', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (seconds)', fontsize=11)
    ax.set_ylabel('Acceleration (g)', fontsize=11)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Show coefficients
    coeff_text = f"Coefficients:\n"
    for i in range(min(4, N+1)):
        coeff_text += f"c_{i}={coeffs[i]:>6.2f}\n"
    if N+1 > 4:
        coeff_text += "..."
    
    ax.text(0.98, 0.5, coeff_text,
            transform=ax.transAxes,
            verticalalignment='center',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
            fontsize=9, family='monospace')

plt.suptitle('Fall Signal Approximation with Legendre Polynomials', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print the coefficients for N=15
print(f"\n📊 Coefficients for N=15 (16 polynomials):")
Phi = np.zeros((n_samples, 16))
for n in range(16):
    Pn = legendre(n)
    Phi[:, n] = Pn(t_norm)
coeffs, _, _, _ = lstsq(Phi, fall_signal)

for i, c in enumerate(coeffs):
    print(f"  c_{i:2d} = {c:>8.4f}  ← Weight for P_{i}(t)")

print(f"\n💡 Key Insight:")
print(f"  Instead of storing 200 samples, we store 16 coefficients!")
print(f"  Compression ratio: {n_samples/16:.1f}x")
print(f"  The coefficients ENCODE the temporal pattern!")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-1, 1, 1000)

P0 = np.ones_like(x)
P1 = x
P2 = (3*x**2 - 1)/2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot P₀
axes[0].plot(x, P0, 'b-', linewidth=2, label='P₀(x) = 1')
axes[0].axhline(0, color='black', linewidth=0.5, alpha=0.3)
axes[0].grid(True, alpha=0.3)
axes[0].set_title('P₀(x) = 1\n(Constant)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].set_ylim(-1.5, 1.5)

# Plot P₁
axes[1].plot(x, P1, 'g-', linewidth=2, label='P₁(x) = x')
axes[1].axhline(0, color='black', linewidth=0.5, alpha=0.3)
axes[1].axvline(0, color='black', linewidth=0.5, alpha=0.3)
axes[1].grid(True, alpha=0.3)
axes[1].set_title('P₁(x) = x\n(Linear)', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].set_ylim(-1.5, 1.5)

# Plot P₂
axes[2].plot(x, P2, 'r-', linewidth=2, label='P₂(x) = (3x²-1)/2')
axes[2].axhline(0, color='black', linewidth=0.5, alpha=0.3)
axes[2].axvline(0, color='black', linewidth=0.5, alpha=0.3)
axes[2].grid(True, alpha=0.3)
axes[2].set_title('P₂(x) = (3x²-1)/2\n(Parabola)', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].set_ylim(-1.5, 1.5)

# Mark key features
axes[2].plot([-1/np.sqrt(3), 1/np.sqrt(3)], [0, 0], 'ro', markersize=8, label='Zeros')
axes[2].plot([0], [-0.5], 'bo', markersize=8, label='Minimum')
axes[2].plot([-1, 1], [1, 1], 'go', markersize=8, label='P₂(±1) = 1')

plt.tight_layout()
plt.show()

# Show orthogonality
print("="*80)
print("VERIFICATION OF ORTHOGONALITY")
print("="*80)

# Numerical integration
from scipy.integrate import quad

integral_P0_P2 = quad(lambda x: 1 * ((3*x**2 - 1)/2), -1, 1)[0]
integral_P1_P2 = quad(lambda x: x * ((3*x**2 - 1)/2), -1, 1)[0]

print(f"\n∫₋₁¹ P₀(x)·P₂(x) dx = {integral_P0_P2:.10f}  (should be 0)")
print(f"∫₋₁¹ P₁(x)·P₂(x) dx = {integral_P1_P2:.10f}  (should be 0)")
print("\n✅ Both are essentially zero (within numerical precision)!")

